made by Kataoka @2024/04/26.  
  
Vertification for Sequence to Sequence Training.  
Target System is a reaction diffusion system.  
  
reference : https://arxiv.org/abs/2109.01050  
github :   https://github.com/a1k12/characterizing-pinns-failure-modes/tree/main

Environment built by Docker, Pytorch Image "23.08-py3".  
https://catalog.ngc.nvidia.com/orgs/nvidia/containers/pytorch

Reaction Diffusion System
$$
\frac{\partial u}{\partial t} - \nu \frac{\partial^2 u}{\partial x^2} - \rho u(1-u) = 0, \quad x \in  \Omega, \quad t \in (0,T]
$$
$$
u(x,0)=h(x)=e^{\frac{(x-\pi)^2}{2(\pi/4)^2}}, \quad x \in  \Omega
$$

In [1]:
# Standard Library

import os
import sys
import copy
import glob
import time
import pickle
import matplotlib.cm
import matplotlib.pyplot as plt

# External Library
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Settings

In [2]:
# Set default dtype to float32
torch.set_default_dtype(torch.float)

# PyTorch random number generator
torch.manual_seed(1234)

# Random number generators in other libraries
np.random.seed(1234)

# Device configuration
# device = torch.device("cpu")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Preprocessing & Visualization

In [3]:
### residual-points sampler

def sample_from_line(
        num_sampling, 
        x_range
    ):
    
    """
    sampling points from line region.
    """
    
    points_array = np.random.rand(num_sampling, 1)

    size_array = np.concatenate([np.full((num_sampling, 1), x_range[1]-x_range[0])], axis=1)
    min_array = np.concatenate([np.full((num_sampling, 1), x_range[0])], axis=1)

    return points_array * size_array + min_array


def sample_from_square(
        num_sampling, 
        x_range, 
        y_range
    ):
    
    """
    sampling points from square region.
    """
    
    points_array = np.random.rand(num_sampling, 2)

    size_array = np.concatenate([np.full((num_sampling, 1), x_range[1]-x_range[0]), np.full((num_sampling, 1), y_range[1]-y_range[0])], axis=1)
    min_array = np.concatenate([np.full((num_sampling, 1), x_range[0]), np.full((num_sampling, 1), y_range[0])], axis=1)

    return points_array * size_array + min_array

In [4]:
def plot_residualPoints(
    points_data
):
    """
    plot residual points
    """
    
    fig = plt.figure(figsize = (6, 4))
    ax = fig.add_subplot(111)
    
    if type(points_data)==dict:
        
        ax.scatter(points_data["initial"][:,1:2], points_data["initial"][:,0:1], s=2.0, label="initial", c="limegreen")
        ax.scatter(np.concatenate([points_data["x_min_boundary"][:,1:2], points_data["x_max_boundary"][:,1:2]]), 
                             np.concatenate([points_data["x_min_boundary"][:,0:1], points_data["x_max_boundary"][:,0:1]]), s=2.0, label="boundary", c="cornflowerblue")
        ax.scatter(points_data["residual"][:,1:2], points_data["residual"][:,0:1], s=2.0, label="residual", c="dimgrey")
        
    elif type(points_data)==list:
        
        for i, data in enumerate(points_data):
            
            if i==0:
                ax.scatter(data["initial"][:,1:2], data["initial"][:,0:1], s=2.0, label="initial", c="limegreen")
                ax.scatter(np.concatenate([data["x_min_boundary"][:,1:2], data["x_max_boundary"][:,1:2]]), 
                                     np.concatenate([data["x_min_boundary"][:,0:1], data["x_max_boundary"][:,0:1]]), s=2.0, label="boundary", c="cornflowerblue")
                ax.scatter(data["residual"][:,1:2], data["residual"][:,0:1], s=2.0, label="residual", c="dimgrey")
                
            else:
                ax.scatter(data["initial"][:,1:2], data["initial"][:,0:1], s=2.0, c="limegreen")
                ax.scatter(np.concatenate([data["x_min_boundary"][:,1:2], data["x_max_boundary"][:,1:2]]), 
                                     np.concatenate([data["x_min_boundary"][:,0:1], data["x_max_boundary"][:,0:1]]), s=2.0, c="cornflowerblue")
                ax.scatter(data["residual"][:,1:2], data["residual"][:,0:1], s=2.0, c="dimgrey")
                
    ax.set_xticks(np.linspace(0, 1.0, 6))
    ax.set_yticks(np.linspace(0, 2*np.pi, 5))
    ax.set_xticklabels(np.round(np.linspace(0.0, 1.0, 6),3))
    ax.set_yticklabels([r"$-\pi$", r"$-\pi/2$",r"$0.0$",r"$\pi/2$",r"$\pi$"])
    
    ax.set_xlabel("t", size = 12)
    ax.set_ylabel("x", size = 12)
    ax.legend(loc="lower right")
    
    return fig

In [5]:
def plotScalar(
    data, 
    label="quantity",
    vmin=0.0,
    vmax=1.0
):
    """
    plot Scalar Distribution
    """
    
    fig = plt.figure(figsize = (7.5, 4))
    ax = fig.add_subplot(111)
    
    Q = ax.imshow(data, aspect="auto", cmap="coolwarm", vmin=vmin, vmax=vmax)
    
    height = data.shape[0]
    width = data.shape[1]
    
    ax.set_xlabel("x", size = 12)
    ax.set_ylabel("y", size = 12)
    
    ax.set_xticks(np.linspace(0, width-1, 6))
    ax.set_yticks(np.linspace(0, height-1, 5))
    ax.set_xticklabels(np.round(np.linspace(0.0, 1.0, 6),3))
    ax.set_yticklabels([r"$-\pi$", r"$-\pi/2$",r"$0.0$",r"$\pi/2$",r"$\pi$"])
    
    fig.colorbar(Q, label=label, shrink=1.0)
    
    return fig

In [6]:
### Simulate analytical solution
### Borrowed from the author's github. TY.

def function(u0: str):
    """Initial condition, string --> function."""

    if u0 == 'sin(x)':
        u0 = lambda x: np.sin(x)
    elif u0 == 'sin(pix)':
        u0 = lambda x: np.sin(np.pi*x)
    elif u0 == 'sin^2(x)':
        u0 = lambda x: np.sin(x)**2
    elif u0 == 'sin(x)cos(x)':
        u0 = lambda x: np.sin(x)*np.cos(x)
    elif u0 == '0.1sin(x)':
        u0 = lambda x: 0.1*np.sin(x)
    elif u0 == '0.5sin(x)':
        u0 = lambda x: 0.5*np.sin(x)
    elif u0 == '10sin(x)':
        u0 = lambda x: 10*np.sin(x)
    elif u0 == '50sin(x)':
        u0 = lambda x: 50*np.sin(x)
    elif u0 == '1+sin(x)':
        u0 = lambda x: 1 + np.sin(x)
    elif u0 == '2+sin(x)':
        u0 = lambda x: 2 + np.sin(x)
    elif u0 == '6+sin(x)':
        u0 = lambda x: 6 + np.sin(x)
    elif u0 == '10+sin(x)':
        u0 = lambda x: 10 + np.sin(x)
    elif u0 == 'sin(2x)':
        u0 = lambda x: np.sin(2*x)
    elif u0 == 'tanh(x)':
        u0 = lambda x: np.tanh(x)
    elif u0 == '2x':
        u0 = lambda x: 2*x
    elif u0 == 'x^2':
        u0 = lambda x: x**2
    elif u0 == 'gauss':
        x0 = np.pi
        sigma = np.pi/4
        u0 = lambda x: np.exp(-np.power((x - x0)/sigma, 2.)/2.)
    return u0


def reaction(u, rho, dt):
    """ du/dt = rho*u*(1-u)
    """
    factor_1 = u * np.exp(rho * dt)
    factor_2 = (1 - u)
    u = factor_1 / (factor_2 + factor_1)
    return u


def diffusion(u, nu, dt, IKX2):
    """ du/dt = nu*d2u/dx2
    """
    factor = np.exp(nu * IKX2 * dt)
    u_hat = np.fft.fft(u)
    u_hat *= factor
    u = np.real(np.fft.ifft(u_hat))
    return u


def reaction_diffusion_discrete_solution(u0 : str, nu, rho, nx = 256, nt = 100):
    """ Computes the discrete solution of the reaction-diffusion PDE using
        pseudo-spectral operator splitting.
    Args:
        u0: initial condition
        nu: diffusion coefficient
        rho: reaction coefficient
        nx: size of x-tgrid
        nt: number of points in the t grid
    Returns:
        u: solution
    """
    L = 2*np.pi
    T = 1
    dx = L/nx
    dt = T/nt
    x = np.arange(0, L, dx) # not inclusive of the last point
    t = np.linspace(0, T, nt).reshape(-1, 1)
    X, T = np.meshgrid(x, t)
    u = np.zeros((nx, nt))

    IKX_pos = 1j * np.arange(0, nx/2+1, 1)
    IKX_neg = 1j * np.arange(-nx/2+1, 0, 1)
    IKX = np.concatenate((IKX_pos, IKX_neg))
    IKX2 = IKX * IKX

    # call u0 this way so array is (n, ), so each row of u should also be (n, )
    u0 = function(u0)
    u0 = u0(x)

    u[:,0] = u0
    u_ = u0
    for i in range(nt-1):
        u_ = reaction(u_, rho, dt)
        u_ = diffusion(u_, nu, dt, IKX2)
        u[:,i+1] = u_

    u = u.T
    #u = u.flatten()
    return u

# Physics-Informed Neural Networks

In [7]:
### Multilayer perceptron

class DNN(nn.Module):
    
    """
    This is Simple Multilayer perceptron class.
    """

    def __init__(self, input_dim, output_dim, num_layer, num_neuron):
        
        super(DNN, self).__init__()
        
        # instance variables
        self.input_size = input_dim
        self.output_size = output_dim
        self.num_layer = num_layer
        self.num_neuron = num_neuron

        # layers
        self.inputs = nn.Linear(input_dim, num_neuron)
        for i in range(1, self.num_layer):
            exec_command = "self.fc" + str(i) + " = nn.Linear(num_neuron, num_neuron)"
            exec(exec_command)
        self.outputs = nn.Linear(num_neuron, output_dim)
        
        # activation function
        self.activation = nn.Tanh()
        
        # weight initialization
        nn.init.xavier_normal_(self.inputs.weight.data, gain=1.0)
        nn.init.zeros_(self.inputs.bias.data)
        nn.init.xavier_normal_(self.outputs.weight.data, gain=1.0)
        nn.init.zeros_(self.outputs.bias.data)

        for i in range(1, self.num_layer):
            exec_command_1 = "nn.init.xavier_normal_(self.fc" + str(i) + ".weight.data, gain=1.0)"
            exec(exec_command_1)
            exec_command_2 = "nn.init.zeros_(self.fc" + str(i) + ".bias.data)"
            exec(exec_command_2)

        
    def forward(self, x):
            
        x = self.activation(self.inputs(x))
        for i in range(1, self.num_layer):
            exec_command = "x = self.activation(self.fc" + str(i) + "(x))"
            exec(exec_command)
        x = self.outputs(x)
        
        return x

In [8]:
### Physics-Informed Neural Networks

class PhysicsInformedNN():
    
    """
    This is Physics-Informed Neural Networks class.
    """
        
    def __init__(self, device, input_size, output_size, num_layer, num_neuron, rho, nu, X, U, opt_params={}):
        
        self.device = device
        
        # Data
        self.x_res = torch.tensor(X["residual"][:, 0:1], requires_grad=True).float()
        self.t_res = torch.tensor(X["residual"][:, 1:2], requires_grad=True).float()
        
        self.x_bound_min = torch.tensor(X["x_min_boundary"][:, 0:1], requires_grad=True).float()
        self.t_bound_min = torch.tensor(X["x_min_boundary"][:, 1:2], requires_grad=True).float()
        
        self.x_bound_max = torch.tensor(X["x_max_boundary"][:, 0:1], requires_grad=True).float()
        self.t_bound_max = torch.tensor(X["x_max_boundary"][:, 1:2], requires_grad=True).float()
        
        self.x_init = torch.tensor(X["initial"][:, 0:1], requires_grad=True).float()
        self.t_init = torch.tensor(X["initial"][:, 1:2], requires_grad=True).float()
        self.u_init = torch.tensor(U["initial"][:, 0:1]).float()
        
        # system parameter
        self.rho = rho
        self.nu = nu
        
        # Loss History
        self.loss_history = {"initial": [], "boundary": [], "residual": [], "total": []}
        
        # epoch information
        self.epoch = 0
        
        # Deep Neural Networks
        self.dnn = DNN(input_size, output_size, num_layer, num_neuron).to(device)
        
        # optimizer
        self.optimizer = torch.optim.LBFGS(self.dnn.parameters(), **opt_params)
        
        
    def net_u(self, x, t):  
        
        outputs = self.dnn(torch.cat([x, t], dim=1).to(self.device))
        u = outputs[:, 0:1]
        
        return u
    
    
    def net_f(self, x, t):

        u = self.net_u(x, t)
        
        u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), retain_graph=True, create_graph=True)[0]
        u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), retain_graph=True, create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), retain_graph=True, create_graph=True)[0]       
        
        f = u_t - self.nu * u_xx - self.rho * u * (1-u)
        
        return f
    
    
    def train(self, nIter):
        
        self.dnn.train()
        
        # Training
        for epoch in range(nIter):
            
            start_time = time.time()
            
            self.epoch += 1

            def closure():
                self.optimizer.zero_grad()
            
                loss, _, _, _ = self.culc_loss()
                loss.backward()
                
                return loss

            self.optimizer.step(closure)

            # calculate the loss again for monitoring
            loss, init_loss, bound_loss, res_loss = self.culc_loss()

            self.loss_history["initial"].append(init_loss.item())
            self.loss_history["boundary"].append(bound_loss.item())
            self.loss_history["residual"].append(res_loss.item())
            self.loss_history["total"].append(loss.item())
            
            end_time = time.time() 
            elapsed_time = end_time - start_time
            
            if self.epoch%100==0:

                print(
                    "Epoch: %d,  time: %.3f,  Loss - init: %.8f,  boundary: %.8f,  residual: %.8f,  total: %.8f" % 
                    (
                        self.epoch,
                        elapsed_time,
                        init_loss.item(),
                        bound_loss.item(), 
                        res_loss.item(), 
                        loss.item()
                    )
                )
            
        
    def culc_loss(self):
        
        x_init = self.x_init.to(self.device)
        t_init = self.t_init.to(self.device)
        u_init = self.u_init.to(self.device)
        
        u_init_pred = self.net_u(x_init, t_init)
        init_loss = torch.mean((u_init - u_init_pred)**2)
        
        x_bound_min = self.x_bound_min.to(self.device)
        t_bound_min = self.t_bound_min.to(self.device)
        x_bound_max = self.x_bound_max.to(self.device)
        t_bound_max = self.t_bound_max.to(self.device)
        
        u_bound_min_pred = self.net_u(x_bound_min, t_bound_min)
        u_bound_max_pred = self.net_u(x_bound_max, t_bound_max)
        bound_loss = torch.mean((u_bound_min_pred - u_bound_max_pred)**2)

        x_res = torch.cat([self.x_res, self.x_init, self.x_bound_min, self.x_bound_max], dim=0).to(self.device)
        t_res = torch.cat([self.t_res, self.t_init, self.t_bound_min, self.t_bound_max], dim=0).to(self.device)
        
        f_pred = self.net_f(x_res, t_res)
        res_loss = torch.mean(f_pred**2)
        
        loss = init_loss + bound_loss + res_loss
        
        return loss, init_loss, bound_loss, res_loss
            
            
    def predict(self, X):
        x = torch.tensor(X[:, 0:1], requires_grad=True).float().to(self.device)
        t = torch.tensor(X[:, 1:2], requires_grad=True).float().to(self.device)
        
        self.dnn.eval()
        u = self.net_u(x, t)
        u = u.detach().cpu().numpy()
        
        return u

# Main

In [9]:
### make points

num_residual = 512
num_boundary = 128
num_initial = 128

points_original = {}    # for original PINNs
points_seq2seq = []    # for seq2seq PINNs

for i in range(0, 10):
    
    start_time = 0.1 * i
    end_time = 0.1 * (i+1)
    
    X = {}

    X["residual"] = sample_from_square(num_residual, [0.0, 2*np.pi], [start_time, end_time])

    t_array = sample_from_line(num_boundary, [start_time, end_time]).reshape(-1, 1)
    x_array = np.full(t_array.shape, 0.0)
    X["x_min_boundary"] = np.concatenate([x_array, t_array], axis=1)

    # t_array = sample_from_line(num_boundary, [start_time, end_time]).reshape(-1, 1)
    x_array = np.full(t_array.shape, 2*np.pi)
    X["x_max_boundary"] = np.concatenate([x_array, t_array], axis=1)

    x_array = sample_from_line(num_initial, [0.0, 2*np.pi]).reshape(-1, 1)
    t_array = np.full(x_array.shape, start_time)
    X["initial"] = np.concatenate([x_array, t_array], axis=1)
    
    if i==0:
        
        points_original = X.copy()
        points_seq2seq.append(X)
        
    else:
        
        for key in X.keys():
            
            if key == "initial":
                pass
            
            else:
                points_original[key] = np.concatenate([points_original[key], X[key]], axis=0)

        points_seq2seq.append(X)

In [10]:
### Teacher Data for initial condition and boundary condition

def initial_h(x):
    
    return np.exp(-(x-np.pi)**2/(2*(np.pi/4)**2))


# boundary condition is periodic boundary condition, so teacher data only need initial condition.

quantity_original = {}
quantity_seq2seq = []

U = {}
U["initial"] = initial_h(points_seq2seq[0]["initial"][:, 0:1])

quantity_original = U.copy()
quantity_seq2seq.append(U)

In [11]:
### analytical solution

u_analytical = reaction_diffusion_discrete_solution("gauss", nu=3.0, rho=5.0, nx=200, nt=500)

fig = plotScalar(u_analytical.T)
# fig.savefig("analytical_solution.png")

# Original PINNs

In [12]:
### parameter

rho = 5.0
nu = 3.0

input_size = 2
output_size = 1
num_layer = 4
num_neuron = 50

In [13]:
###optimizer parameters

opt_params = {"lr": 0.1}

In [14]:
### build a model

model = PhysicsInformedNN(device, input_size, output_size, num_layer, num_neuron, rho, nu, points_original, quantity_original, opt_params)

In [15]:
### training Model

numIter = 1000

model.train(numIter)

In [16]:
### result

x = np.linspace(0.0, 2*np.pi, 200)
t = np.linspace(0.0, 1.0, 500)
xxx, ttt = np.meshgrid(x, t)

temp = np.concatenate([xxx.reshape((-1,1)),ttt.reshape((-1,1))], axis=1)
u_original = model.predict(temp).reshape(500, 200)

print("Relative Error : {}".format(np.round(np.abs((u_original-u_analytical)/u_analytical).mean(),8)))
print("Absolute Error : {}".format(np.round(np.abs(u_original-u_analytical).mean(),8)))

fig = plot_residualPoints(points_original)
# fig.savefig("points_originalPINNs.png")

fig = plotScalar(u_original.T)
# fig.savefig("prediction_originalPINNs.png")

fig = plotScalar(np.abs(u_original-u_analytical).T, vmax=np.abs(u_original-u_analytical).max())
# fig.savefig("points_originalPINNs.png")

# seq2seq PINNs

In [17]:
### parameter

rho = 5.0
nu = 3.0

input_size = 2
output_size = 1
num_layer = 4
num_neuron = 50

In [18]:
###optimizer parameters

opt_params = {"lr": 0.1}

In [19]:
### build a model

models = [PhysicsInformedNN(device, input_size, output_size, num_layer, num_neuron, rho, nu, points_seq2seq[0], quantity_seq2seq[0], opt_params)]

In [20]:
### training Model

numIter = 200

for i in range(0, 10):
    
    if i==0:
        models[i].train(2*numIter)
        
    else:
        nextU = {}
        nextU["initial"] = models[i-1].predict(points_seq2seq[i]["initial"])
        quantity_seq2seq.append(nextU)
        
        models.append(PhysicsInformedNN(device, input_size, output_size, num_layer, num_neuron, rho, nu, points_seq2seq[i], quantity_seq2seq[i], opt_params))
        models[i].train(numIter)

In [21]:
for i in range(10):

    t = 0.1 * i
    x = np.linspace(0.0, 2*np.pi, 200)
    t = np.linspace(t, 0.1*(i+1), 50)
    xxx, ttt = np.meshgrid(x, t)

    temp = np.concatenate([xxx.reshape((-1,1)),ttt.reshape((-1,1))], axis=1)
    u_temp = models[i].predict(temp)
    
    if i==0:
        u_seq2seq = u_temp.reshape(50,200)
    else:
        u_seq2seq = np.concatenate([u_seq2seq, u_temp.reshape(50,200)], axis=0)

print("Relative Error : {}".format(np.round(np.abs((u_seq2seq-u_analytical)/u_analytical).mean(),8)))
print("Absolute Error : {}".format(np.round(np.abs(u_seq2seq-u_analytical).mean(),8)))

u_seq2seq

fig = plot_residualPoints(points_seq2seq)
# fig.savefig("points_seq2seqPINNs.png")

fig = plotScalar(u_seq2seq.T)
# fig.savefig("prediction_originalPINNs.png")

fig = plotScalar(np.abs(u_seq2seq-u_analytical).T, vmax=np.abs(u_seq2seq-u_analytical).max())
# fig.savefig("points_originalPINNs.png")